In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import meow as mw

## 1. Define the rib waveguide geometry

In [ ]:
# Parameters (matching emepy_diverging)
wl = 1.0
widths = [0.8, 1.0]
t_slab = 0.2
t_soi = 1.5
t_ox = 0.2
n_Si = 3.4
n_SiO2 = 1.4
w_sim = 3.0
h_sim = 2.5
y_bot = -0.5

si = mw.IndexMaterial(n=n_Si, name="Si", meta={"color": "orange"})
sio2 = mw.IndexMaterial(n=n_SiO2, name="SiO2", meta={"color": "steelblue"})

env = mw.Environment(wl=wl, T=25.0)
mesh = mw.Mesh2D(
    x=np.linspace(-w_sim / 2, w_sim / 2, 101),
    y=np.linspace(y_bot, y_bot + h_sim, 101),
    num_pml=(10, 10),
)


def make_cross_section(w):
    """Build a CrossSection for a rib waveguide of width w."""
    structures = [
        # SiO2 base layer (below slab + oxide)
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=y_bot,
                y_max=t_slab + t_ox,
            ),
            mesh_order=2,
        ),
        # SiO2 cladding around ridge
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w / 2 - t_ox / 2,
                x_max=w / 2 + t_ox / 2,
                y_min=t_slab + t_ox,
                y_max=t_soi + t_ox,
            ),
            mesh_order=2,
        ),
        # Si slab (full width)
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=0.0,
                y_max=t_slab,
            ),
            mesh_order=1,
        ),
        # Si ridge
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w / 2,
                x_max=w / 2,
                y_min=t_slab,
                y_max=t_soi,
            ),
            mesh_order=1,
        ),
    ]
    return mw.CrossSection(structures=structures, mesh=mesh, env=env)


cs_left = make_cross_section(widths[0])
cs_right = make_cross_section(widths[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cs_left._visualize(ax=axes[0], show=False)
axes[0].set_title(f"Left (w={widths[0]} um)")
cs_right._visualize(ax=axes[1], show=False)
axes[1].set_title(f"Right (w={widths[1]} um)")
plt.tight_layout()
plt.show()

## 2. Compute and visualize modes for both cross-sections

In [ ]:
# Compute modes for both cross-sections
num_modes = 100
modes_left_raw = mw.compute_modes(cs_left, num_modes=num_modes)
modes_right_raw = mw.compute_modes(cs_right, num_modes=num_modes)

modes_left = mw.orthonormalize(modes_left_raw)
modes_right = mw.orthonormalize(modes_right_raw)

print(
    f"Computed {len(modes_left_raw)} left/{len(modes_right_raw)} right raw modes; "
    f"kept {len(modes_left)} left/{len(modes_right)} right after orthonormalization"
)

In [ ]:
# Visualize left and right mode sets
print("Left cross-section modes")
mw.visualize(modes_left[:2], fields=("Ex", "Hx"))

print("Right cross-section modes")
mw.visualize(modes_right[:2], fields=("Ex", "Hx"))

In [ ]:
# Determine orthonormality convention from overlaps


def overlap_matrix(modes, op):
    n = len(modes)
    M = np.zeros((n, n), dtype=np.complex128)
    for i, mi in enumerate(modes):
        for j, mj in enumerate(modes):
            M[i, j] = op(mi, mj)
    return M


def report_orthonormality(name, modes):
    I = np.eye(len(modes), dtype=np.complex128)
    M_dot = overlap_matrix(modes, lambda a, b: mw.inner_product(a, b))
    M_dot_conj = overlap_matrix(
        modes, lambda a, b: mw.inner_product(a, b, conjugate=True)
    )

    err_dot = np.linalg.norm(M_dot - I)
    err_dot_conj = np.linalg.norm(M_dot_conj - I)

    print(f"\n{name}")
    print(f"||dot - I||_F      = {err_dot:.3e}")
    print(f"||dot_conj - I||_F = {err_dot_conj:.3e}")

    if err_dot < err_dot_conj:
        print("Orthonormal for: dot")
    elif err_dot_conj < err_dot:
        print("Orthonormal for: dot_conj")
    else:
        print("Orthonormal for: neither (or tie)")

    return M_dot, M_dot_conj


M_left_dot, M_left_dot_conj = report_orthonormality("Left cross-section", modes_left)  # fmt: skip
M_right_dot, M_right_dot_conj = report_orthonormality("Right cross-section", modes_right)  # fmt: skip